In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 4 — Pipeline Orchestrator
# MAGIC Runs Bronze → Silver → Gold in sequence.
# MAGIC
# MAGIC | Mode | When to use |
# MAGIC |------|-------------|
# MAGIC | `full_refresh` | First run, or full rebuild |
# MAGIC | `incremental`  | Weekly run — only new Year+WeekNumber rows |
# MAGIC
# MAGIC ### Databricks Job setup
# MAGIC ```
# MAGIC Workflows → Jobs → Create Job
# MAGIC   Task       : Notebook  →  /path/to/04_orchestrator
# MAGIC   Schedule   : 0 6 * * 1   (every Monday 6 AM)
# MAGIC   Parameters : {"mode": "incremental"}
# MAGIC ```

# COMMAND ----------
# MAGIC %md ## 0. Configuration + Mode

# COMMAND ----------

dbutils.widgets.text("mode", "full_refresh")
MODE = dbutils.widgets.get("mode").strip().lower()

if MODE not in ["full_refresh", "incremental"]:
    raise ValueError("mode must be 'full_refresh' or 'incremental'")

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

current_user = spark.sql("SELECT current_user()").collect()[0][0]

CATALOG = "workspace"
SCHEMA  = "promotion_sql"
RAW_PATH =f"/Volumes/workspace/default/course_data/Promotion_raw_data"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")
print(f"Raw path: {RAW_PATH}")
print(f"Mode    : {MODE}")
print(f"Run ID  : {RUN_ID}")


# COMMAND ----------
# MAGIC %md ## 1. Watermark + Run Log

# COMMAND ----------

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.pipeline_watermarks")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.pipeline_run_log")

spark.sql(f"""
CREATE TABLE {CATALOG}.{SCHEMA}.pipeline_watermarks (
    table_name      STRING,
    last_year       INT,
    last_week       INT,
    rows_processed  BIGINT,
    run_status      STRING,
    run_mode        STRING,
    updated_at      TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE {CATALOG}.{SCHEMA}.pipeline_run_log (
    run_id       STRING,
    run_mode     STRING,
    stage        STRING,
    rows_out     BIGINT,
    status       STRING,
    error_msg    STRING,
    started_at   TIMESTAMP,
    finished_at  TIMESTAMP
)
USING DELTA
""")

def get_watermark():
    rows = spark.sql(f"""
        SELECT last_year, last_week
        FROM {CATALOG}.{SCHEMA}.pipeline_watermarks
        WHERE table_name = 'bronze_sales'
          AND run_status = 'SUCCESS'
        ORDER BY updated_at DESC
        LIMIT 1
    """).collect()
    return (int(rows[0]["last_year"]), int(rows[0]["last_week"])) if rows else (1900, 0)

def set_watermark(year, week, n, status="SUCCESS"):
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.pipeline_watermarks 
        (table_name, last_year, last_week, rows_processed, run_status, run_mode, updated_at)
        VALUES (
            'bronze_sales',
            {year},
            {week},
            {n},
            '{status}',
            '{MODE}',
            CURRENT_TIMESTAMP()
        )
    """)
    print(f"  Watermark → Year={year}, Week={week} ({n:,} rows, {status})")

def log(stage, rows=0, status="SUCCESS", error=""):
    safe_error = (error or "")[:200].replace("'", "")
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.pipeline_run_log 
        (run_id, run_mode, stage, rows_out, status, error_msg, started_at, finished_at)
        VALUES (
            '{RUN_ID}',
            '{MODE}',
            '{stage}',
            {rows},
            '{status}',
            '{safe_error}',
            CURRENT_TIMESTAMP(),
            CURRENT_TIMESTAMP()
        )
    """)


# COMMAND ----------
# MAGIC %md ## 2. Shared helpers

# COMMAND ----------

def read_raw(filename):
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(f"{RAW_PATH}/{filename}")
    )

def table_exists(full_name: str) -> bool:
    try:
        spark.table(full_name)
        return True
    except Exception:
        return False


# COMMAND ----------
# MAGIC %md ## 3. Bronze Stage

# COMMAND ----------

def run_bronze():
    print("\n▓▓▓  BRONZE  ▓▓▓")
    t = datetime.now()

    def ingest(filename, table, mode="overwrite"):
        full_name = f"{CATALOG}.{SCHEMA}.bronze_{table}"

        df = read_raw(filename)
        df = (
            df.withColumn("_ingest_time", F.current_timestamp())
              .withColumn("_source_file", F.col("_metadata.file_path"))
              .withColumn("_batch_date", F.current_date())
        )

        (
            df.write
            .format("delta")
            .mode(mode)
            .option("overwriteSchema", "true")
            .saveAsTable(full_name)
        )

        n = spark.table(full_name).count()
        print(f"  {full_name:45s} {n:>8,} rows")
        return n

    # Dimensions: always full overwrite
    ingest("RAW_Product.csv",   "product", mode="overwrite")
    ingest("RAW_Store.csv",     "store", mode="overwrite")
    ingest("RAW_Date.csv",      "date", mode="overwrite")
    ingest("RAW_Promotion.csv", "promotion", mode="overwrite")

    # Fact: incremental on Year + WeekNumber
    full_name = f"{CATALOG}.{SCHEMA}.bronze_sales"

    df_raw = (
        read_raw("RAW_Sales.csv")
        .withColumn("_y", F.col("Year").cast(IntegerType()))
        .withColumn("_w", F.col("WeekNumber").cast(IntegerType()))
    )

    if MODE == "incremental":
        wm_year, wm_week = get_watermark()
        print(f"  Watermark: Year={wm_year}, Week={wm_week}")

        df_new = df_raw.filter(
            (F.col("_y") > wm_year) |
            ((F.col("_y") == wm_year) & (F.col("_w") > wm_week))
        )
        n = df_new.count()

        if n == 0:
            print("  bronze_sales                                  0 new rows — skipping")
            log("bronze", rows=0)
            return

        write_mode = "append"
    else:
        df_new = df_raw
        n = df_new.count()
        write_mode = "overwrite"

    df_out = (
        df_new.drop("_y", "_w")
              .withColumn("_ingest_time", F.current_timestamp())
              .withColumn("_source_file", F.col("_metadata.file_path"))
              .withColumn("_batch_date", F.current_date())
    )

    (
        df_out.write
        .format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )

    max_row = df_new.orderBy(F.desc("_y"), F.desc("_w")).first()
    set_watermark(int(max_row["_y"]), int(max_row["_w"]), n)

    print(f"  {full_name:45s} {n:>8,} rows")

    log("bronze", rows=n)
    print(f"  ✔ {(datetime.now() - t).seconds}s")


# COMMAND ----------
# MAGIC %md ## 4. Silver Stage

# COMMAND ----------

def run_silver():
    print("\n▓▓▓  SILVER  ▓▓▓")
    t = datetime.now()

    # ── dim_product ────────────────────────────────────────────────
    spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.silver_dim_product
    USING DELTA AS
    WITH cleaned_product AS (
        SELECT
            INITCAP(TRIM(Product)) AS Product,
            INITCAP(TRIM(Brand)) AS Brand,
            INITCAP(TRIM(Category)) AS Category,
            TRIM(Size) AS Size,
            INITCAP(TRIM(Supplier)) AS Supplier,
            CAST(UnitCost AS DOUBLE) AS UnitCost,
            ROW_NUMBER() OVER (PARTITION BY TRIM(Product) ORDER BY TRIM(Product)) AS rn
        FROM {CATALOG}.{SCHEMA}.bronze_product
    )
    SELECT
        ROW_NUMBER() OVER (ORDER BY Product) AS ProductKey,
        Product,
        Brand,
        Category,
        Size,
        Supplier,
        UnitCost
    FROM cleaned_product
    WHERE rn = 1
    ORDER BY Product
    """)
    n1 = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.silver_dim_product").collect()[0][0]
    print(f"  {CATALOG}.{SCHEMA}.silver_dim_product{' ' * (45 - len(f'{CATALOG}.{SCHEMA}.silver_dim_product'))} {n1:>8,} rows")

    # ── dim_store ──────────────────────────────────────────────────
    spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.silver_dim_store
    USING DELTA AS
    WITH cleaned_store AS (
        SELECT
            TRIM(StoreID) AS StoreID,
            INITCAP(TRIM(StoreName)) AS StoreName,
            INITCAP(TRIM(City)) AS City,
            INITCAP(TRIM(Province)) AS Province,
            UPPER(TRIM(ProvinceAbbrev)) AS ProvinceAbbrev,
            INITCAP(TRIM(Country)) AS Country,
            ROW_NUMBER() OVER (PARTITION BY TRIM(StoreID) ORDER BY TRIM(StoreID)) AS rn
        FROM {CATALOG}.{SCHEMA}.bronze_store
    )
    SELECT
        ROW_NUMBER() OVER (ORDER BY StoreID) AS StoreKey,
        StoreID,
        StoreName,
        City,
        Province,
        ProvinceAbbrev,
        Country
    FROM cleaned_store
    WHERE rn = 1
    ORDER BY StoreID
    """)
    n2 = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.silver_dim_store").collect()[0][0]
    print(f"  {CATALOG}.{SCHEMA}.silver_dim_store{' ' * (45 - len(f'{CATALOG}.{SCHEMA}.silver_dim_store'))} {n2:>8,} rows")

    # ── dim_date ───────────────────────────────────────────────────
    spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.silver_dim_date
    USING DELTA AS
    WITH cleaned_date AS (
        SELECT
            CAST(Year AS INT) AS Year,
            CAST(WeekNumber AS INT) AS WeekNumber,
            CAST(MonthNumber AS INT) AS MonthNumber,
            CONCAT('FY', REGEXP_EXTRACT(FiscalYear, '(\\d{{4}})', 1)) AS FiscalYear,
            INITCAP(TRIM(Month)) AS Month,
            WeekStartDate,
            CASE
                WHEN Quarter IS NOT NULL THEN Quarter
                WHEN MonthNumber IN (2, 3, 4) THEN 'Q1'
                WHEN MonthNumber IN (5, 6, 7) THEN 'Q2'
                WHEN MonthNumber IN (8, 9, 10) THEN 'Q3'
                ELSE 'Q4'
            END AS Quarter,
            ROW_NUMBER() OVER (PARTITION BY CAST(Year AS INT), CAST(WeekNumber AS INT) ORDER BY CAST(Year AS INT), CAST(WeekNumber AS INT)) AS rn
        FROM {CATALOG}.{SCHEMA}.bronze_date
    )
    SELECT
        CAST(Year * 100 + WeekNumber AS INT) AS DateKey,
        Year,
        WeekNumber,
        WeekStartDate,
        Month,
        MonthNumber,
        Quarter,
        FiscalYear
    FROM cleaned_date
    WHERE rn = 1
    ORDER BY Year, WeekNumber
    """)
    n3 = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.silver_dim_date").collect()[0][0]
    print(f"  {CATALOG}.{SCHEMA}.silver_dim_date{' ' * (45 - len(f'{CATALOG}.{SCHEMA}.silver_dim_date'))} {n3:>8,} rows")

    # ── dim_promotion ──────────────────────────────────────────────
    spark.sql(rf"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.silver_dim_promotion
    USING DELTA AS
    WITH cleaned_promotion AS (
        SELECT
            TRIM(PromotionName) AS PromotionName,
            INITCAP(TRIM(OnFlyer)) AS OnFlyer,
            CASE
                WHEN TRIM(Discount) LIKE '%\%' THEN
                    TRY_CAST(REGEXP_EXTRACT(TRIM(Discount), '([\d\.]+)', 1) AS DOUBLE) / 100
                WHEN TRY_CAST(Discount AS DOUBLE) > 1 THEN
                    TRY_CAST(Discount AS DOUBLE) / 100
                ELSE
                    TRY_CAST(Discount AS DOUBLE)
            END AS Discount,
            INITCAP(TRIM(PromotionType)) AS PromotionType,
            CASE
                WHEN TRIM(DiscountTier) IS NOT NULL AND TRIM(DiscountTier) != '' THEN INITCAP(TRIM(DiscountTier))
                WHEN TRY_CAST(Discount AS DOUBLE) >= 0.30 THEN 'Deep'
                WHEN TRY_CAST(Discount AS DOUBLE) > 0.00 THEN 'Mid'
                ELSE 'None'
            END AS DiscountTier,
            ROW_NUMBER() OVER (PARTITION BY TRIM(PromotionName) ORDER BY TRIM(PromotionName)) AS rn
        FROM {CATALOG}.{SCHEMA}.bronze_promotion
    )
    SELECT
        ROW_NUMBER() OVER (ORDER BY PromotionName) AS PromotionKey,
        PromotionName,
        OnFlyer,
        Discount,
        PromotionType,
        DiscountTier
    FROM cleaned_promotion
    WHERE rn = 1
    ORDER BY PromotionName
    """)
    n4 = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.silver_dim_promotion").collect()[0][0]
    print(f"  {CATALOG}.{SCHEMA}.silver_dim_promotion{' ' * (45 - len(f'{CATALOG}.{SCHEMA}.silver_dim_promotion'))} {n4:>8,} rows")

    # ── fact_sales ─────────────────────────────────────────────────
    if MODE == "incremental" and table_exists(f"{CATALOG}.{SCHEMA}.silver_fact_sales"):
        spark.sql(rf"""
        MERGE INTO {CATALOG}.{SCHEMA}.silver_fact_sales AS target
        USING (
            WITH cleaned_sales AS (
                SELECT
                    INITCAP(TRIM(Product)) AS Product,
                    INITCAP(TRIM(OnFlyer)) AS OnFlyer,
                    CASE
                        WHEN TRIM(Discount) LIKE '%\%' THEN
                            TRY_CAST(REGEXP_EXTRACT(TRIM(Discount), '([\d\.]+)', 1) AS DOUBLE) / 100
                        WHEN TRY_CAST(Discount AS DOUBLE) > 1 THEN
                            TRY_CAST(Discount AS DOUBLE) / 100
                        ELSE
                            TRY_CAST(Discount AS DOUBLE)
                    END AS Discount,
                    CAST(Year AS INT) AS Year,
                    CAST(WeekNumber AS INT) AS WeekNumber,
                    TRIM(StoreID) AS StoreID,
                    CAST(Price AS DOUBLE) AS Price,
                    CAST(Units AS INT) AS Units,
                    CAST(SalesAmt AS DOUBLE) AS SalesAmt,
                    CAST(GrossMargin AS DOUBLE) AS GrossMargin,
                    CAST(Transactions AS INT) AS Transactions,
                    ROW_NUMBER() OVER (PARTITION BY CAST(Year AS INT), CAST(WeekNumber AS INT), TRIM(StoreID), INITCAP(TRIM(Product)) 
                                       ORDER BY CAST(Year AS INT), CAST(WeekNumber AS INT)) AS rn
                FROM {CATALOG}.{SCHEMA}.bronze_sales
            ),
            with_promotion_name AS (
                SELECT
                    *,
                    CASE
                        WHEN Discount = 0 THEN 'No Promotion'
                        WHEN OnFlyer = 'Yes' THEN CONCAT(CAST(Discount * 100 AS INT), '% Off + Flyer')
                        ELSE CONCAT(CAST(Discount * 100 AS INT), '% Off')
                    END AS PromotionName
                FROM cleaned_sales
                WHERE rn = 1
            ),
            joined AS (
                SELECT
                    p.ProductKey,
                    s.StoreKey,
                    d.DateKey,
                    pr.PromotionKey,
                    f.Year,
                    f.WeekNumber,
                    COALESCE(f.Price, ROUND(f.SalesAmt / f.Units, 2)) AS ActualPrice,
                    f.Discount AS DiscountPct,
                    f.Units AS UnitsSold,
                    f.SalesAmt,
                    f.GrossMargin,
                    f.Transactions,
                    CASE WHEN f.GrossMargin < 0 THEN 1 ELSE 0 END AS IsBelowCost
                FROM with_promotion_name f
                LEFT JOIN {CATALOG}.{SCHEMA}.silver_dim_product p ON f.Product = p.Product
                LEFT JOIN {CATALOG}.{SCHEMA}.silver_dim_store s ON f.StoreID = s.StoreID
                LEFT JOIN {CATALOG}.{SCHEMA}.silver_dim_date d ON f.Year = d.Year AND f.WeekNumber = d.WeekNumber
                LEFT JOIN {CATALOG}.{SCHEMA}.silver_dim_promotion pr ON f.PromotionName = pr.PromotionName
            )
            SELECT * FROM joined
        ) AS source
        ON target.ProductKey = source.ProductKey 
           AND target.StoreKey = source.StoreKey 
           AND target.DateKey = source.DateKey 
           AND target.PromotionKey = source.PromotionKey
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        """)
    else:
        spark.sql(rf"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.silver_fact_sales
        USING DELTA AS
        WITH cleaned_sales AS (
            SELECT
                INITCAP(TRIM(Product)) AS Product,
                INITCAP(TRIM(OnFlyer)) AS OnFlyer,
                CASE
                    WHEN TRIM(Discount) LIKE '%\%' THEN
                        TRY_CAST(REGEXP_EXTRACT(TRIM(Discount), '([\d\.]+)', 1) AS DOUBLE) / 100
                    WHEN TRY_CAST(Discount AS DOUBLE) > 1 THEN
                        TRY_CAST(Discount AS DOUBLE) / 100
                    ELSE
                        TRY_CAST(Discount AS DOUBLE)
                END AS Discount,
                CAST(Year AS INT) AS Year,
                CAST(WeekNumber AS INT) AS WeekNumber,
                TRIM(StoreID) AS StoreID,
                CAST(Price AS DOUBLE) AS Price,
                CAST(Units AS INT) AS Units,
                CAST(SalesAmt AS DOUBLE) AS SalesAmt,
                CAST(GrossMargin AS DOUBLE) AS GrossMargin,
                CAST(Transactions AS INT) AS Transactions,
                ROW_NUMBER() OVER (PARTITION BY CAST(Year AS INT), CAST(WeekNumber AS INT), TRIM(StoreID), INITCAP(TRIM(Product)) 
                                   ORDER BY CAST(Year AS INT), CAST(WeekNumber AS INT)) AS rn
            FROM {CATALOG}.{SCHEMA}.bronze_sales
        ),
        with_promotion_name AS (
            SELECT
                *,
                CASE
                    WHEN Discount = 0 THEN 'No Promotion'
                    WHEN OnFlyer = 'Yes' THEN CONCAT(CAST(Discount * 100 AS INT), '% Off + Flyer')
                    ELSE CONCAT(CAST(Discount * 100 AS INT), '% Off')
                END AS PromotionName
            FROM cleaned_sales
            WHERE rn = 1
        )
        SELECT
            p.ProductKey,
            s.StoreKey,
            d.DateKey,
            pr.PromotionKey,
            f.Year,
            f.WeekNumber,
            COALESCE(f.Price, ROUND(f.SalesAmt / f.Units, 2)) AS ActualPrice,
            f.Discount AS DiscountPct,
            f.Units AS UnitsSold,
            f.SalesAmt,
            f.GrossMargin,
            f.Transactions,
            CASE WHEN f.GrossMargin < 0 THEN 1 ELSE 0 END AS IsBelowCost
        FROM with_promotion_name f
        LEFT JOIN {CATALOG}.{SCHEMA}.silver_dim_product p ON f.Product = p.Product
        LEFT JOIN {CATALOG}.{SCHEMA}.silver_dim_store s ON f.StoreID = s.StoreID
        LEFT JOIN {CATALOG}.{SCHEMA}.silver_dim_date d ON f.Year = d.Year AND f.WeekNumber = d.WeekNumber
        LEFT JOIN {CATALOG}.{SCHEMA}.silver_dim_promotion pr ON f.PromotionName = pr.PromotionName
        """)

    n5 = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.silver_fact_sales").collect()[0][0]
    print(f"  {CATALOG}.{SCHEMA}.silver_fact_sales{' ' * (45 - len(f'{CATALOG}.{SCHEMA}.silver_fact_sales'))} {n5:>8,} rows")

    log("silver", rows=n5)
    print(f"  ✔ {(datetime.now() - t).seconds}s")


# COMMAND ----------
# MAGIC %md ## 5. Gold Stage

# COMMAND ----------

def run_gold():
    print("\n▓▓▓  GOLD  ▓▓▓")
    t = datetime.now()

    # ── gold_price_elasticity ──────────────────────────────────────
    spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.gold_price_elasticity
    USING DELTA AS
    WITH weekly_agg AS (
        SELECT
            f.ProductKey,
            p.Product,
            p.UnitCost,
            f.PromotionKey,
            pr.PromotionName,
            pr.OnFlyer,
            pr.DiscountTier,
            f.ActualPrice,
            f.DiscountPct,
            f.Year,
            f.WeekNumber,
            SUM(f.UnitsSold) AS ChainUnits,
            SUM(f.SalesAmt) AS ChainSalesAmt,
            SUM(f.GrossMargin) AS ChainGrossMargin,
            SUM(f.Transactions) AS ChainTransactions,
            SUM(f.IsBelowCost) AS StoresBelowCost,
            COUNT(f.StoreKey) AS StoresActive
        FROM {CATALOG}.{SCHEMA}.silver_fact_sales f
        JOIN {CATALOG}.{SCHEMA}.silver_dim_product p ON f.ProductKey = p.ProductKey
        JOIN {CATALOG}.{SCHEMA}.silver_dim_promotion pr ON f.PromotionKey = pr.PromotionKey
        GROUP BY
            f.ProductKey, p.Product, p.UnitCost,
            f.PromotionKey, pr.PromotionName, pr.OnFlyer, pr.DiscountTier,
            f.ActualPrice, f.DiscountPct, f.Year, f.WeekNumber
    )
    SELECT
        ProductKey,
        Product,
        UnitCost,
        PromotionName,
        OnFlyer,
        DiscountTier,
        ActualPrice,
        DiscountPct,
        COUNT(WeekNumber) AS WeeksAtThisPrice,
        CAST(ROUND(AVG(ChainUnits), 0) AS BIGINT) AS AvgWeeklyUnits,
        CAST(ROUND(SUM(ChainUnits), 0) AS BIGINT) AS TotalUnits,
        ROUND(AVG(ChainSalesAmt), 2) AS AvgWeeklySales,
        ROUND(SUM(ChainSalesAmt), 2) AS TotalSales,
        ROUND(AVG(ChainGrossMargin), 2) AS AvgWeeklyMargin,
        ROUND(SUM(ChainGrossMargin), 2) AS TotalMargin,
        CAST(ROUND(AVG(ChainTransactions), 0) AS BIGINT) AS AvgWeeklyTransactions,
        CAST(ROUND(AVG(StoresActive), 0) AS BIGINT) AS AvgStoresActive,
        ROUND(SUM(ChainGrossMargin) / SUM(ChainSalesAmt) * 100, 1) AS GrossMarginPct,
        ROUND(AVG(ChainGrossMargin) / AVG(ChainUnits), 4) AS MarginPerUnit,
        RANK() OVER (PARTITION BY Product ORDER BY AVG(ChainUnits) DESC) AS RankByUnits,
        RANK() OVER (PARTITION BY Product ORDER BY AVG(ChainGrossMargin) DESC) AS RankByMargin,
        CURRENT_TIMESTAMP() AS _gold_ts
    FROM weekly_agg
    GROUP BY
        ProductKey, Product, UnitCost,
        PromotionName, OnFlyer, DiscountTier,
        ActualPrice, DiscountPct
    ORDER BY Product, ActualPrice
    """)
    n1 = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.gold_price_elasticity").collect()[0][0]
    print(f"  {CATALOG}.{SCHEMA}.gold_price_elasticity{' ' * (45 - len(f'{CATALOG}.{SCHEMA}.gold_price_elasticity'))} {n1:>8,} rows")

    # ── gold_flyer_impact ──────────────────────────────────────────
    spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.gold_flyer_impact
    USING DELTA AS
    WITH flyer_sales AS (
        SELECT
            pr.OnFlyer,
            pr.DiscountTier,
            COUNT(DISTINCT f.ProductKey) AS ProductsOffered,
            COUNT(DISTINCT CONCAT(f.Year, '-', f.WeekNumber)) AS WeeksActive,
            CAST(ROUND(SUM(f.UnitsSold), 0) AS BIGINT) AS TotalUnits,
            ROUND(SUM(f.SalesAmt), 2) AS TotalSales,
            ROUND(SUM(f.GrossMargin), 2) AS TotalMargin,
            CAST(ROUND(SUM(f.Transactions), 0) AS BIGINT) AS TotalTransactions,
            ROUND(AVG(f.ActualPrice), 2) AS AvgPrice,
            ROUND(AVG(f.DiscountPct), 4) AS AvgDiscountPct
        FROM {CATALOG}.{SCHEMA}.silver_fact_sales f
        JOIN {CATALOG}.{SCHEMA}.silver_dim_promotion pr ON f.PromotionKey = pr.PromotionKey
        GROUP BY pr.OnFlyer, pr.DiscountTier
    )
    SELECT
        OnFlyer,
        DiscountTier,
        ProductsOffered,
        WeeksActive,
        TotalUnits,
        TotalSales,
        TotalMargin,
        TotalTransactions,
        AvgPrice,
        AvgDiscountPct,
        ROUND(TotalMargin / TotalSales * 100, 1) AS GrossMarginPct,
        ROUND(TotalUnits / WeeksActive, 0) AS AvgWeeklyUnits,
        ROUND(TotalSales / WeeksActive, 2) AS AvgWeeklySales,
        ROUND(TotalMargin / TotalUnits, 4) AS MarginPerUnit,
        CURRENT_TIMESTAMP() AS _gold_ts
    FROM flyer_sales
    ORDER BY OnFlyer DESC, DiscountTier
    """)
    n2 = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.gold_flyer_impact").collect()[0][0]
    print(f"  {CATALOG}.{SCHEMA}.gold_flyer_impact{' ' * (45 - len(f'{CATALOG}.{SCHEMA}.gold_flyer_impact'))} {n2:>8,} rows")

    # ── gold_store_performance ─────────────────────────────────────
    spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.gold_store_performance
    USING DELTA AS
    SELECT
        s.StoreKey,
        s.StoreID,
        s.StoreName,
        s.City,
        s.Province,
        s.Country,
        COUNT(DISTINCT CONCAT(f.Year, '-', f.WeekNumber)) AS WeeksActive,
        COUNT(DISTINCT f.ProductKey) AS ProductsSold,
        CAST(ROUND(SUM(f.UnitsSold), 0) AS BIGINT) AS TotalUnits,
        ROUND(SUM(f.SalesAmt), 2) AS TotalSales,
        ROUND(SUM(f.GrossMargin), 2) AS TotalMargin,
        CAST(ROUND(SUM(f.Transactions), 0) AS BIGINT) AS TotalTransactions,
        ROUND(AVG(f.ActualPrice), 2) AS AvgPrice,
        ROUND(SUM(f.GrossMargin) / SUM(f.SalesAmt) * 100, 1) AS GrossMarginPct,
        ROUND(SUM(f.UnitsSold) / COUNT(DISTINCT CONCAT(f.Year, '-', f.WeekNumber)), 0) AS AvgWeeklyUnits,
        ROUND(SUM(f.SalesAmt) / COUNT(DISTINCT CONCAT(f.Year, '-', f.WeekNumber)), 2) AS AvgWeeklySales,
        SUM(f.IsBelowCost) AS WeeksBelowCost,
        CURRENT_TIMESTAMP() AS _gold_ts
    FROM {CATALOG}.{SCHEMA}.silver_fact_sales f
    JOIN {CATALOG}.{SCHEMA}.silver_dim_store s ON f.StoreKey = s.StoreKey
    GROUP BY
        s.StoreKey, s.StoreID, s.StoreName, s.City, s.Province, s.Country
    ORDER BY TotalSales DESC
    """)
    n3 = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.gold_store_performance").collect()[0][0]
    print(f"  {CATALOG}.{SCHEMA}.gold_store_performance{' ' * (45 - len(f'{CATALOG}.{SCHEMA}.gold_store_performance'))} {n3:>8,} rows")

    log("gold", rows=(n1 + n2 + n3))
    print(f"  ✔ {(datetime.now() - t).seconds}s")


# COMMAND ----------
# MAGIC %md ## 6. Run Pipeline

# COMMAND ----------

t_pipeline = datetime.now()
print(f"\n{'='*60}")
print(f"  PIPELINE START: {t_pipeline.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Mode: {MODE}")
print(f"{'='*60}")

try:
    run_bronze()
except Exception as e:
    log("bronze", status="FAILED", error=str(e))
    raise

try:
    run_silver()
except Exception as e:
    log("silver", status="FAILED", error=str(e))
    raise

try:
    run_gold()
except Exception as e:
    log("gold", status="FAILED", error=str(e))
    raise

t_end = datetime.now()
duration = (t_end - t_pipeline).seconds
print(f"\n{'='*60}")
print(f"  PIPELINE COMPLETE: {t_end.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Duration: {duration}s")
print(f"{'='*60}\n")